# TCGA Data Download → Google Drive
**Person 1 (Bojana) — Run this in Colab**

Downloads 800 TCGA samples directly to Google Drive. No GPU needed.

**Runtime → Run all**

In [ ]:
# Step 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2 — Install dependencies
!pip install -q requests pandas tqdm

In [ ]:
# Step 3 — Configuration
# UPDATE THIS to your shared Google Drive folder name
SHARED_FOLDER_NAME = "multiomics-project"

from pathlib import Path
DRIVE_ROOT   = Path(f"/content/drive/MyDrive/{SHARED_FOLDER_NAME}")
DATA_DIR     = DRIVE_ROOT / "data/raw"
MANIFEST_DIR = DRIVE_ROOT / "data/manifests"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving to: {DRIVE_ROOT}")

In [ ]:
# Step 4 — Download all 800 samples
import requests
import json
import pandas as pd
from tqdm import tqdm

PROJECTS = ["TCGA-BRCA", "TCGA-LUAD", "TCGA-COAD", "TCGA-KIRC", "TCGA-LIHC", "TCGA-THCA"]
PROJECT_SAMPLE_LIMITS = {
    "TCGA-BRCA": 134,
    "TCGA-LUAD": 134,
    "TCGA-COAD": 133,
    "TCGA-KIRC": 133,
    "TCGA-LIHC": 133,
    "TCGA-THCA": 133,
}
FILES_ENDPOINT = "https://api.gdc.cancer.gov/files"
DATA_ENDPOINT  = "https://api.gdc.cancer.gov/data"

def get_files(filters):
    params = {
        "filters": json.dumps(filters),
        "fields": "file_id,file_name,cases.submitter_id,cases.case_id",
        "format": "JSON",
        "size": "2000"
    }
    r = requests.get(FILES_ENDPOINT, params=params)
    r.raise_for_status()
    return r.json()["data"]["hits"]

def query_matched_samples(project_id):
    rna_filter = {"op": "and", "content": [
        {"op": "in", "content": {"field": "cases.project.project_id", "value": [project_id]}},
        {"op": "in", "content": {"field": "files.data_type", "value": ["Gene Expression Quantification"]}},
        {"op": "in", "content": {"field": "files.analysis.workflow_type", "value": ["STAR - Counts"]}}
    ]}
    meth_filter = {"op": "and", "content": [
        {"op": "in", "content": {"field": "cases.project.project_id", "value": [project_id]}},
        {"op": "in", "content": {"field": "files.data_type", "value": ["Methylation Beta Value"]}},
        {"op": "in", "content": {"field": "files.platform", "value": ["Illumina Human Methylation 450"]}}
    ]}
    rna_files  = get_files(rna_filter)
    meth_files = get_files(meth_filter)
    rna_map    = {h["cases"][0]["case_id"]: h for h in rna_files  if h.get("cases")}
    meth_map   = {h["cases"][0]["case_id"]: h for h in meth_files if h.get("cases")}
    common     = sorted(set(rna_map.keys()) & set(meth_map.keys()))
    limit      = PROJECT_SAMPLE_LIMITS[project_id]
    pairs = []
    for case_id in common[:limit]:
        pairs.append({
            "project":       project_id,
            "case_id":       case_id,
            "rna_file_id":   rna_map[case_id]["file_id"],
            "rna_file_name": rna_map[case_id]["file_name"],
            "meth_file_id":  meth_map[case_id]["file_id"],
            "meth_file_name":meth_map[case_id]["file_name"],
        })
    return pairs

def download_file(file_id, dest_path):
    if dest_path.exists():
        return  # already downloaded, skip
    r = requests.post(
        DATA_ENDPOINT,
        data=json.dumps({"ids": [file_id]}),
        headers={"Content-Type": "application/json"},
        stream=True
    )
    r.raise_for_status()
    temp = dest_path.with_suffix(dest_path.suffix + ".part")
    with open(temp, "wb") as f:
        for chunk in r.iter_content(chunk_size=65536):
            if chunk:
                f.write(chunk)
    temp.replace(dest_path)

# Query all projects
print("Querying GDC API...")
all_matched = []
for project in PROJECTS:
    pairs = query_matched_samples(project)
    all_matched.extend(pairs)
    print(f"  {project}: {len(pairs)} matched samples")

# Save manifest
manifest_path = MANIFEST_DIR / "matched_samples.csv"
pd.DataFrame(all_matched).to_csv(manifest_path, index=False)
print(f"\nManifest saved: {manifest_path} ({len(all_matched)} samples)")

# Download files
print("\nStarting downloads...")
for project in PROJECTS:
    pairs = [p for p in all_matched if p["project"] == project]
    proj_dir = DATA_DIR / project
    (proj_dir / "rna_seq").mkdir(parents=True, exist_ok=True)
    (proj_dir / "methylation").mkdir(parents=True, exist_ok=True)

    print(f"\n{project} ({len(pairs)} samples)")
    for pair in tqdm(pairs):
        download_file(pair["rna_file_id"],  proj_dir / "rna_seq"    / pair["rna_file_name"])
        download_file(pair["meth_file_id"], proj_dir / "methylation" / pair["meth_file_name"])

print("\nAll downloads complete!")

In [ ]:
# Step 5 — Verify
for project in PROJECTS:
    rna  = list((DATA_DIR / project / "rna_seq").glob("*.tsv"))
    meth = list((DATA_DIR / project / "methylation").glob("*.txt"))
    print(f"{project}: RNA={len(rna)}  Methylation={len(meth)}")